In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.datasets import ImageFolder
import math

In [ ]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else 'cpu'
print(device)

cuda


In [ ]:
class LinearFunction(torch.autograd.Function) :

  @staticmethod
  def forward(ctx, input, weight, bias=None) :
    ctx.save_for_backward(input, weight, bias)

    output = F.linear(input, weight, bias)
    return output

  @staticmethod
  def backward(ctx, grad_output):
    input, weight, bias = ctx.saved_tensors

    grad_input = grad_output @ weight
    grad_weight = grad_output.T @ input
    grad_bias = torch.sum(grad_output, dim=0)


    return grad_input, grad_weight, grad_bias

In [ ]:
class ConvolutionFunction(torch.autograd.Function) :

  @staticmethod
  def forward(ctx, input, weight, bias=None, stride = 1, padding = 0) :
    ctx.save_for_backward(input, weight, bias)
    ctx.stride = stride
    ctx.padding = padding

    output = F.conv2d(input, weight, bias, stride, padding)
    return output

  @staticmethod
  def backward(ctx, grad_output) :
    input, weight, bias = ctx.saved_tensors
    stride = ctx.stride
    padding = ctx.padding

    grad_input = nn.grad.conv2d_input(input.shape, weight, grad_output, stride, padding)
    grad_weight = nn.grad.conv2d_weight(input, weight.shape, grad_output, stride, padding)
    grad_bias = torch.sum(grad_output, dim=(0,2,3))

    return grad_input, grad_weight, grad_bias, None, None

In [ ]:
class Dense(nn.Module):

  def __init__(self, input_size, output_size):
    super(Dense, self).__init__()
    self.weights = nn.Parameter(torch.empty(output_size, input_size, device=device))
    self.bias = nn.Parameter(torch.zeros(output_size, device=device))
    nn.init.kaiming_uniform_(self.weights, a=math.sqrt(5))

  def forward(self, input) :
    return LinearFunction.apply(input, self.weights, self.bias)

In [ ]:
class Convolution(nn.Module) :

  def __init__(self, in_channel, out_channel, kernel_size) :
    super(Convolution, self).__init__()
    self.kernels = nn.Parameter(torch.empty(out_channel, in_channel, kernel_size, kernel_size, device=device))
    self.bias = nn.Parameter(torch.zeros(out_channel, device=device))
    nn.init.kaiming_uniform_(self.kernels, a=math.sqrt(5))

  def forward(self, input) :
    return ConvolutionFunction.apply(input, self.kernels, self.bias)

In [ ]:
class MaxPool(nn.Module) :
  def __init__(self, kernel_size) :
    super(MaxPool, self).__init__()
    self.kernel_size = kernel_size

  def forward(self, input) :
    output = F.max_pool2d(input, self.kernel_size)
    return output

In [ ]:
transform = Compose([
    Resize((256,256)),
    ToTensor(),
    Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [ ]:
train_data_dir = "./drive/MyDrive/tomato/train"
train_data = ImageFolder(train_data_dir, transform=transform)

test_data_dir = "./drive/MyDrive/tomato/val"
test_data = ImageFolder(test_data_dir, transform=transform)

In [ ]:
print(len(train_data.classes))
image , label = train_data[0]
print(image.shape, train_data.classes[label])

10
torch.Size([3, 256, 256]) Tomato___Bacterial_spot


In [ ]:
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

In [ ]:
class NeuralNet(nn.Module) :

  def __init__(self):
    super(NeuralNet, self).__init__()

    self.conv1 = Convolution(3, 32, 3)
    self.conv2 = Convolution(32, 64, 3)
    self.conv3 = Convolution(64, 128, 3)
    self.conv4 = Convolution(128, 256, 3)
    self.conv5 = Convolution(256, 512, 3)
    self.pool = MaxPool(2)

    self.fc1 = Dense(6*6*512,1024)
    self.fc2 = Dense(1024, 64)
    self.fc3 = Dense(64, 10)
    self.flatten = nn.Flatten(start_dim=1)

    # self.conv1 = nn.Conv2d(3,32,3)
    # self.conv2 = nn.Conv2d(32, 64, 3)
    # self.conv3 = nn.Conv2d(64, 128, 3)
    # self.conv4 = nn.Conv2d(128, 256, 3)
    # self.conv5 = nn.Conv2d(256, 512, 3)
    # self.pool = nn.MaxPool2d(2)

    # self.fc1 = nn.Linear(6*6*512,1024)
    # self.fc2 = nn.Linear(1024, 64)
    # self.fc3 = nn.Linear(64, 10)
    # self.flatten = nn.Flatten(start_dim=1)

  def forward(self, x) :
    x = self.pool(F.relu(self.conv1(x)))
    x = self.pool(F.relu(self.conv2(x)))
    x = self.pool(F.relu(self.conv3(x)))
    x = self.pool(F.relu(self.conv4(x)))
    x = self.pool(F.relu(self.conv5(x)))

    x = self.flatten(x)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

In [ ]:
model = NeuralNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(model.parameters)
criterion = nn.CrossEntropyLoss()

<bound method Module.parameters of NeuralNet(
  (conv1): Convolution()
  (conv2): Convolution()
  (conv3): Convolution()
  (conv4): Convolution()
  (conv5): Convolution()
  (pool): MaxPool()
  (fc1): Dense()
  (fc2): Dense()
  (fc3): Dense()
  (flatten): Flatten(start_dim=1, end_dim=-1)
)>


In [ ]:
print(len(train_loader))

313


In [ ]:
for epoch in range(30) :

  print(f"Epoch : {epoch} ")
  running_loss = 0.0
  for i, data in enumerate(train_loader):
    images, labels = data
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs,labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item()

    # if (i+1) % 50 == 0:
    #   print(f"Batch : {i+1} Loss : {running_loss}")

  print(f"Running Loss : {running_loss/len(train_loader)}")

Epoch : 0 
Running Loss : 1.5997069160016582
Epoch : 1 
Running Loss : 0.9677917187968001
Epoch : 2 
Running Loss : 0.6951690269544863
Epoch : 3 
Running Loss : 0.5430285916827358
Epoch : 4 
Running Loss : 0.42193372497638576
Epoch : 5 
Running Loss : 0.3298808992646944
Epoch : 6 
Running Loss : 0.2559836762079511
Epoch : 7 
Running Loss : 0.20211566541903317
Epoch : 8 
Running Loss : 0.1566875502407646
Epoch : 9 
Running Loss : 0.1273623185303693
Epoch : 10 
Running Loss : 0.10047905186351877
Epoch : 11 
Running Loss : 0.08106202113043409
Epoch : 12 
Running Loss : 0.07041750437324902
Epoch : 13 
Running Loss : 0.06131324420973938
Epoch : 14 
Running Loss : 0.05995891698150369
Epoch : 15 
Running Loss : 0.0785116545596389
Epoch : 16 
Running Loss : 0.04474011288744549
Epoch : 17 
Running Loss : 0.07254573507811882
Epoch : 18 
Running Loss : 0.03569454263099626
Epoch : 19 
Running Loss : 0.052599677308808455
Epoch : 20 
Running Loss : 0.052410871152140825
Epoch : 21 
Running Loss : 0.0

In [ ]:
torch.save(model.state_dict(), 'tomato.pth')

In [ ]:
from sklearn.metrics import confusion_matrix,recall_score,precision_score,accuracy_score
import numpy as np


In [ ]:
tomato_classifier = NeuralNet().to(device)
tomato_classifier.load_state_dict(torch.load('tomato.pth'))

<All keys matched successfully>

In [ ]:
tomato_classifier.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = tomato_classifier(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

print("Recall:", recall_score(all_labels, all_preds, average='macro'))
print("Precision:", precision_score(all_labels, all_preds, average='macro'))
print("Accuracy:", accuracy_score(all_labels, all_preds))

Confusion Matrix:
[[ 99   0   1   0   0   0   0   0   0   0]
 [  1  81   8   1   4   4   0   1   0   0]
 [  1   5  88   3   0   2   0   1   0   0]
 [  0   0   0  92   3   1   2   0   0   2]
 [  1   2   3   5  85   4   0   0   0   0]
 [  0   0   1   0   4  94   0   0   1   0]
 [  0   4   0   0   6   5  77   0   2   6]
 [  1   1   0   0   0   3   0  95   0   0]
 [  0   0   0   0   0   0   0   0 100   0]
 [  0   0   2   0   0   0   0   0   0  98]]
Recall: 0.909
Precision: 0.9112051627937223
Accuracy: 0.909


In [ ]:
from PIL import Image

In [ ]:
random_image = Image.open("./test.jpeg")
random_image = transform(random_image).unsqueeze(0).to(device)
_, prediction = torch.max(tomato_classifier(random_image), 1)
print(train_data.classes[prediction.item()])

Tomato___healthy
